In [3]:
import os
import torch
from transformers import (AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer)
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from tqdm.notebook import tqdm
from dotenv import load_dotenv
import json

d:\project\TraceCall\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on...: {device} ({torch.cuda.get_device_name(0)})")

Running on...: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)


In [5]:
from huggingface_hub import login
load_dotenv()
token = os.getenv("HF_TOKEN")

login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
with open('dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

texts = [item['text'] for item in data]
raw_labels = [item['label'] for item in data]

label2id = {"ไม่พบ": 0, "พบ": 1, "ข้อมูลไม่พอ": 2}
id2label = {0: "ไม่พบ", 1: "พบ", 2: "ข้อมูลไม่พอ"}

labels = [label2id[l]for l in raw_labels]
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels)

print(f"Text for train: {len(train_texts)}")
print(f"Text for validation: {len(val_texts)}")

Text for train: 8000
Text for validation: 2000


## Load model

In [7]:
model_name = "airesearch/wangchanberta-base-att-spm-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

sentences = []

for item in data:
    sentences.append(item['text'])

sample_sen = texts[:3]
inputs = tokenizer(sample_sen, padding=True, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

for text, pred_id, prob in zip(sentences, predictions, probs):
    pred_label = id2label[pred_id.item()]
    conf = prob[pred_id].item() * 100
    print(f"Text: '{text}")
    print(f"Predicted: '{pred_label}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 663.24it/s]
[transformers] CamembertForSequenceClassification LOAD REPORT from: airesearch/wangchanberta-base-att-spm-uncased
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task

Text: 'สำหรับรายละเอียดเรื่องเบี้ยประกัน เดี๋ยวเจ้าหน้าที่ส่งข้อความยืนยันทาง SMS ให้อีกทีครับ
Predicted: 'ข้อมูลไม่พอ
Text: 'เอกสารกรมธรรม์ตัวจริงจะจัดส่งถึงบ้านภายใน 7-14 วันทำการครับ
Predicted: 'ข้อมูลไม่พอ
Text: 'มอบเงินปันผลคืนทุกสิ้นปีกรมธรรม์ ปีละ 800 บาทครับ
Predicted: 'ข้อมูลไม่พอ


## Fine tune

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

class TraceCallDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])

        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TraceCallDataset(train_encodings, train_labels)
val_dataset = TraceCallDataset(val_encodings, val_labels)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
def predict_tracecall(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncations=True, padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]
        pred_id = torch.argmax(probs).item()

    label = id2label[pred_id]
    confidence = probs[pred_id].item() * 100
    print(f"Text: {sentence}")
    print(f"Model predict: {label} | confidence {confidence:.2f}%\n")

predict_tracecall("พี่ลดเบี้ยให้เหลือสองร้อยบาทนะคะ สนใจไหม")
predict_tracecall("พูดอะไรนะ [สัญญาณไม่ดีเลย] ฟังไม่รู้เรื่อง")
predict_tracecall("เบี้ยประกันตัวนี้ราคาไม่แพงนะจ๊ะ")
predict_tracecall("วงเงินคุ้มครองชีวิต 5 ล้านบาทถ้วน")
predict_tracecall("ลูกค้าจ่ายเพียงวันละ 10 บาท หรือเดือนละ 300 บาทเท่านั้นครับ")

Text: พี่ลดเบี้ยให้เหลือสองร้อยบาทนะคะ สนใจไหม
Model predict: ไม่พบ | confidence 86.61%

Text: พูดอะไรนะ [สัญญาณไม่ดีเลย] ฟังไม่รู้เรื่อง
Model predict: ข้อมูลไม่พอ | confidence 99.99%

Text: เบี้ยประกันตัวนี้ราคาไม่แพงนะจ๊ะ
Model predict: ไม่พบ | confidence 99.63%

Text: วงเงินคุ้มครองชีวิต 5 ล้านบาทถ้วน
Model predict: ไม่พบ | confidence 99.98%

Text: ลูกค้าจ่ายเพียงวันละ 10 บาท หรือเดือนละ 300 บาทเท่านั้นครับ
Model predict: พบ | confidence 100.00%



## Call id group up

In [ ]:
calls = {}

for item in data:
    cid = item['callid']
    if cid not in calls:
        calls[cid] = []
    calls[cid].append(item)

NameError: name 'data' is not defined